In [1]:
import sys
print(sys.executable)

C:\Users\sande\AppData\Local\Programs\Python\Python310\python.exe


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("FraudDetection") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [3]:
df = spark.read.csv("creditcard.csv", header=True, inferSchema=True)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [4]:
df.show(5)

+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+-----+
|Time|                V1|                 V2|              V3|                V4|                 V5|                 V6|                 V7|                V8|                V9|                V10|               V11|               V12|               V13|               V14|               V15|               V16|               V17|                V18|               V19|                V20|                 V21|                V22|     

In [5]:
df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [6]:
df.groupBy("Class").count().show()

+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [7]:
df.count()

284807

In [8]:
from pyspark.sql.functions import col, sum

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|   0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|     0|    0|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [9]:
feature_columns = df.columns[:-1]  # all except 'Class'

print(feature_columns)

['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [10]:
import sys
!{sys.executable} -m pip install numpy

     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     --------------------------------------- 0.0/12.9 MB 653.6 kB/s eta 0:00:20
     ---------------------------------------- 0.1/12.9 MB 1.1 MB/s eta 0:00:13
      --------------------------------------- 0.2/12.9 MB 1.6 MB/s eta 0:00:09
      --------------------------------------- 0.2/12.9 MB 1.4 MB/s eta 0:00:10
     - -------------------------------------- 0.4/12.9 MB 2.0 MB/s eta 0:00:07
     - -------------------------------------- 0.5/12.9 MB 1.9 MB/s eta 0:00:07
     -- ------------------------------------- 0.7/12.9 MB 2.3 MB/s eta 0:00:06
     -- ------------------------------------- 0.8/12.9 MB 2.2 MB/s eta 0:00:06
     -- ------------------------------------- 0.9/12.9 MB 2.2 MB/s eta 0:00:06
     --- ------------------------------------ 1.0/12.9 MB 2.2 MB/s eta 0:00:06
     --- ------------------------------------ 1.2/12.9 MB 2.4 MB/s eta 0:00:05
     ---- ----------------------------------- 1.3/12.9 MB 


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\sande\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [11]:
import numpy
print(numpy.__version__)

2.2.6


In [12]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df = assembler.transform(df)

In [13]:
df.select("features", "Class").show(5)

+--------------------+-----+
|            features|Class|
+--------------------+-----+
|[0.0,-1.359807133...|    0|
|[0.0,1.1918571113...|    0|
|[1.0,-1.358354061...|    0|
|[1.0,-0.966271711...|    0|
|[2.0,-1.158233093...|    0|
+--------------------+-----+
only showing top 5 rows



In [14]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Train count:", train_df.count())
print("Test count:", test_df.count())

Train count: 228045
Test count: 56762


In [15]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="Class"
)

model = lr.fit(train_df)

In [16]:
predictions = model.transform(test_df)

In [17]:
predictions.select("Class", "prediction", "probability").show(5)

+-----+----------+--------------------+
|Class|prediction|         probability|
+-----+----------+--------------------+
|    0|       0.0|[0.99976086148174...|
|    0|       0.0|[0.99939282196421...|
|    0|       0.0|[0.99958710850869...|
|    0|       0.0|[0.99871498970585...|
|    0|       0.0|[0.99997518311804...|
+-----+----------+--------------------+
only showing top 5 rows



In [18]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="Class",
    rawPredictionCol="rawPrediction"
)

accuracy = evaluator.evaluate(predictions)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.9627677591855215


In [19]:
predictions.groupBy("Class", "prediction").count().show()

+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    1|       0.0|   32|
|    0|       0.0|56662|
|    1|       1.0|   60|
|    0|       1.0|    8|
+-----+----------+-----+



In [20]:
fraud_count = df.filter(df.Class == 1).count()
normal_count = df.filter(df.Class == 0).count()

ratio = normal_count / fraud_count

print("Ratio:", ratio)

Ratio: 577.8760162601626


In [21]:
from pyspark.sql.functions import when

df = df.withColumn(
    "weight",
    when(df.Class == 1, ratio).otherwise(1)
)

In [22]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

In [23]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="Class",
    weightCol="weight"
)

model = lr.fit(train_df)

In [24]:
predictions = model.transform(test_df)

In [25]:
predictions.groupBy("Class", "prediction").count().show()

+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    1|       0.0|    9|
|    0|       0.0|55330|
|    1|       1.0|   83|
|    0|       1.0| 1340|
+-----+----------+-----+



In [26]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="Class",
    rawPredictionCol="rawPrediction"
)

auc = evaluator.evaluate(predictions)

print("New AUC:", auc)

New AUC: 0.9812345117806371


In [28]:
import sys
!{sys.executable} -m pip install pandas

     ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
     --------------------------------------- 0.0/11.3 MB 991.0 kB/s eta 0:00:12
     ---------------------------------------- 0.1/11.3 MB 1.3 MB/s eta 0:00:09
      --------------------------------------- 0.2/11.3 MB 1.3 MB/s eta 0:00:09
      --------------------------------------- 0.3/11.3 MB 1.5 MB/s eta 0:00:08
      --------------------------------------- 0.3/11.3 MB 1.5 MB/s eta 0:00:08
      --------------------------------------- 0.3/11.3 MB 1.5 MB/s eta 0:00:08
      --------------------------------------- 0.3/11.3 MB 1.5 MB/s eta 0:00:08
      --------------------------------------- 0.3/11.3 MB 1.5 MB/s eta 0:00:08
      --------------------------------------- 0.3/11.3 MB 1.5 MB/s eta 0:00:08
      --------------------------------------- 0.3/11.3 MB 1.5 MB/s eta 0:00:08
      --------------------------------------- 0.3/11.3 MB 1.5 MB/s eta 0:00:08
      --------------------------------------- 0.3/11.3 MB 


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\sande\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [29]:
predictions.select("Class", "prediction", "Amount") \
    .toPandas() \
    .to_csv("fraud_results.csv", index=False)

In [30]:
import os

path = os.path.expanduser("~/Desktop/fraud_results.csv")

predictions.select("Class", "prediction", "Amount") \
    .toPandas() \
    .to_csv(path, index=False)

print("Saved at:", path)

Saved at: C:\Users\sande/Desktop/fraud_results.csv
